<a href="https://colab.research.google.com/github/Namik05/ERP_SIG_Attractif/blob/main/Projet%20Machine%20learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q diffusers transformers accelerate safetensors
!pip install -q imageio imageio-ffmpeg


In [ ]:
import torch
from diffusers import DiffusionPipeline
import imageio


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [ ]:
pipe = DiffusionPipeline.from_pretrained(
    "damo-vilab/text-to-video-ms-1.7b",
    torch_dtype=torch.float16,
    variant="fp16"
)

pipe = pipe.to("cuda")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/787 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/755 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/657 [00:00<?, ?B/s]

vae/diffusion_pytorch_model.fp16.safeten(…):   0%|          | 0.00/167M [00:00<?, ?B/s]

unet/diffusion_pytorch_model.fp16.safete(…):   0%|          | 0.00/2.82G [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!
The TextToVideoSDPipeline has been deprecated and will not receive bug fixes or feature updates after Diffusers version 0.33.1. 


In [ ]:
prompt = "A futuristic city at night with flying cars"

video_frames = pipe(
    prompt,
    num_inference_steps=25,
    num_frames=16
).frames

  0%|          | 0/25 [00:00<?, ?it/s]

In [ ]:
import numpy as np



In [ ]:
def preprocess_frames(frames):
    processed = []
    for frame in frames:
        # Convert tensor → numpy si nécessaire
        if hasattr(frame, "cpu"):
            frame = frame.cpu().numpy()

        # Si format (C, H, W) → (H, W, C)
        if frame.shape[0] == 3:
            frame = np.transpose(frame, (1, 2, 0))

        # Convertir float [0,1] → uint8 [0,255]
        frame = (frame * 255).clip(0, 255).astype(np.uint8)

        processed.append(frame)
    return processed


In [ ]:
video_frames_uint8 = preprocess_frames(video_frames)


In [ ]:
type(video_frames), len(video_frames)


(numpy.ndarray, 1)

In [ ]:
video_frames.shape


(1, 16, 256, 256, 3)

In [ ]:
frames = video_frames[0]   # shape: (num_frames, ...)


In [ ]:
import numpy as np

processed_frames = []

for frame in frames:

    # Si (C, H, W) → (H, W, C)
    if frame.ndim == 3 and frame.shape[0] in [1, 3]:
        frame = np.transpose(frame, (1, 2, 0))

    # Si grayscale → RGB
    if frame.ndim == 2:
        frame = np.stack([frame]*3, axis=-1)

    # Si float → uint8
    if frame.dtype != np.uint8:
        frame = (frame * 255).clip(0, 255).astype(np.uint8)

    # Sécurité finale
    if frame.ndim == 3 and frame.shape[2] == 3:
        processed_frames.append(frame)

print("Nombre de frames prêtes :", len(processed_frames))


Nombre de frames prêtes : 16


In [ ]:
import imageio

video_path = "result_video.mp4"

imageio.mimsave(
    video_path,
    processed_frames,
    fps=8,
    codec="libx264"
)

video_path


'result_video.mp4'

In [ ]:
processed_frames[0].shape, processed_frames[0].dtype


((256, 256, 3), dtype('uint8'))

In [ ]:
from IPython.display import Video

Video("result_video.mp4", embed=True)
